# Taller · De los datos a un dashboard publicado

**Programación — ADMI 1301**

Trabajarás con las **2.000 ventas de una cadena de papelerías en 2025**. En este taller vas a:

1. **Graficar** (6 ejercicios con `groupby`, `crosstab` y `pivot_table`).
2. **Organizar** tus gráficos en una sola figura con `subplots`: tu *dashboard*.
3. **Embellecerlo con prompts**: le pides a Gemini que reescriba el código de tu figura y pegas su respuesta aquí.
4. **Elegir tu versión favorita**, escribir tus conclusiones y **publicar tu notebook en GitHub**.

Las instrucciones completas están en la guía en Word. Aquí encuentras lo esencial de cada ejercicio y una **verificación** (`assert`) para saber si vas bien.

**Escriba los nombres de los integrantes aca.**
______________________________________________________________________  
 1.   
 2.   
 3.    
    


## 0. Preparación
Ejecuta la celda y sube el archivo **`ventas_papelerias_2025.csv`**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display

subidos = files.upload()   # selecciona ventas_papelerias_2025.csv

df = pd.read_csv("/ventas_papelerias_2025 (1).csv")
df["fecha"] = pd.to_datetime(df["fecha"])
df["mes"] = df["fecha"].dt.month        # 1 = enero ... 12 = diciembre
df.head()

**Conoce tus datos** antes de graficar: cuántas filas hay, qué columnas, qué valores tiene cada categoría.

In [ ]:
print(df.shape)
df.info()
for col in ["ciudad", "canal", "segmento", "categoria", "metodo_pago"]:
    print(col, "->", sorted(df[col].unique()))

---
# Parte A · Seis gráficos
En cada ejercicio: **(1)** prepara una tabla con `groupby`, `crosstab` o `pivot_table`, **(2)** guárdala en la variable indicada, **(3)** grafícala. Ponle **título** a cada gráfico.

### Ejercicio 1 · Top 5 de productos *(barras horizontales)*
**Contexto.** La gerente de compras debe decidir en qué productos concentrar el inventario del próximo año.

**Tarea.** Calcula los ingresos (`total`) por `producto`, quédate con los **5 mayores** y grafícalos en barras horizontales. Guarda la tabla en **`top_productos`**.

*Herramientas:* `groupby`, `sum`, `sort_values`, `head`, `.plot.barh()`

In [ ]:
# Ejercicio 1

top_productos = (
    df.groupby("producto")["total"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)


ax = top_productos.sort_values(ascending=True).plot.barh(
    figsize=(8, 4),
    title="Top 5 de productos por ingresos totales"
)
ax.set_xlabel("Ingresos totales")
ax.set_ylabel("Producto")
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Verificación
assert len(top_productos) == 5, "Debe haber exactamente 5 productos"
assert top_productos.index[0] == "Cuaderno", "El producto con más ingresos no coincide: revisa el ordenamiento"
assert abs(top_productos.iloc[0] - 20712345) < 1, "Revisa que estés sumando la columna total"
print("Ejercicio 1 correcto ✔")

### Ejercicio 2 · Ingresos mes a mes: Bogotá vs Medellín *(líneas)*
**Contexto.** La dirección regional quiere comparar la evolución de sus dos ciudades principales durante el año.

**Tarea.** **Selecciona** solo las ventas de Bogotá y Medellín (`isin`). Con una **tabla dinámica** pon los meses en las filas, las ciudades en las columnas y la suma de `total` como valores. Guárdala en **`ing_mes_ciudad`** y grafícala como líneas con marcadores.

*Herramientas:* filtrado con `isin`, `pivot_table`, `.plot.line(marker="o")`

In [ ]:
# Ejercicio 2
# 1. Filtrar solo Bogotá y Medellín
df_ciudades = df[df["ciudad"].isin(["Bogotá", "Medellín"])]

# 2. Crear tabla dinámica (meses en filas, ciudades en columnas, suma de total en valores)
ing_mes_ciudad = df_ciudades.pivot_table(
    index="mes",
    columns="ciudad",
    values="total",
    aggfunc="sum"
)

# 3. Graficar como líneas con marcadores y título
ax = ing_mes_ciudad.plot.line(
    marker="o",
    figsize=(8, 4),
    title="Ingresos mes a mes: Bogotá vs Medellín"
)
ax.set_xlabel("Mes")
ax.set_ylabel("Ingresos totales")
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Verificación
assert ing_mes_ciudad.shape == (12, 2), f"Se esperaba una tabla de 12 meses x 2 ciudades, tienes {ing_mes_ciudad.shape}"
assert set(ing_mes_ciudad.columns) == {"Bogotá", "Medellín"}, "Las columnas deben ser las dos ciudades"
assert abs(ing_mes_ciudad["Bogotá"].sum() - 37847614) < 1, "El total de Bogotá no coincide"
print("Ejercicio 2 correcto ✔")

### Ejercicio 3 · ¿Por qué canal se vende más? *(gráfico de pie)*
**Contexto.** La empresa evalúa si invierte más en su tienda online o en la atención telefónica.

**Tarea.** Calcula los ingresos totales por `canal`, guárdalos en **`ing_canal`** y grafica la participación de cada canal con un gráfico de pie que muestre los **porcentajes con un decimal**.

*Herramientas:* `groupby`, `.plot.pie(autopct=...)`

In [ ]:
# Ejercicio 3
ing_canal = df.groupby("canal")["total"].sum()
ing_canal_grafico = ing_canal.plot.pie(
    autopct="%1.1f%%",
    figsize=(6,6),
    title="Distribución ingresos por canal",
    ylabel=""
)
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Verificación
assert abs(ing_canal.sum() - 118668012) < 1, "La suma de los canales debe ser el ingreso total"
assert abs(ing_canal["Telefónico"] / ing_canal.sum() - 0.520) < 0.01, "La participación del canal Telefónico no coincide"
print("Ejercicio 3 correcto ✔")

### Ejercicio 4 · ¿Cada segmento compra lo mismo? *(barras apiladas)*
**Contexto.** El área comercial sospecha que los clientes corporativos compran productos distintos a los minoristas.

**Tarea.** Con `pd.crosstab` cuenta las ventas por `segmento` (filas) y `categoria` (columnas), **normalizadas por fila** y expresadas en **porcentaje** (de 0 a 100). Guarda la tabla en **`tabla_seg_cat`** y grafícala en barras **apiladas**.

*Herramientas:* `pd.crosstab(..., normalize="index")`, `.plot.bar(stacked=True)`

In [ ]:
# Ejercicio 4
tabla_seg_cat = pd.crosstab(df["segmento"], df["categoria"], normalize="index") * 100
tabla_seg_cat.plot.bar(stacked=True, figsize=(8, 4), title="Porcentaje de ventas por segmento y categoría")
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Verificación
assert tabla_seg_cat.shape == (3, 5), f"Se esperaba 3 segmentos x 5 categorías, tienes {tabla_seg_cat.shape}"
assert (abs(tabla_seg_cat.sum(axis=1) - 100) < 0.01).all(), "Cada fila debe sumar 100 (usa normalize y multiplica por 100)"
print("Ejercicio 4 correcto ✔")

### Ejercicio 5 · Productividad de los vendedores *(dispersión)*
**Contexto.** Recursos humanos revisa qué vendedores concretan más ventas y cuáles venden tickets más altos.

**Tarea.** Con `groupby` por `vendedor` y `ciudad`, calcula para cada vendedor el **número de ventas** (`n_ventas`) y el **valor promedio de sus ventas** (`ticket_prom`). Guarda la tabla en **`vend`** (con índice normal: usa `reset_index()`). Haz una dispersión (`n_ventas` en X, `ticket_prom` en Y) con un **color por ciudad** y leyenda.

*Herramientas:* `groupby().agg(nombre=("columna", "función"))`, `reset_index`, `plt.scatter` dentro de un `for`

In [ ]:
# Ejercicio 5
vend = df.groupby(["vendedor", "ciudad"]).agg(n_ventas=("id_venta", "count"), ticket_prom=("total", "mean")).reset_index()
for ciudad in vend["ciudad"].unique():
    vend_ciudad = vend[vend["ciudad"] == ciudad]
    plt.scatter(vend_ciudad["n_ventas"], vend_ciudad["ticket_prom"], label=ciudad)
plt.legend()


In [ ]:
# ✅ Verificación
assert vend.shape[0] == 12, f"Debe haber una fila por vendedor (12), tienes {vend.shape[0]}"
assert {"vendedor", "ciudad", "n_ventas", "ticket_prom"} <= set(vend.columns), "Faltan columnas: vendedor, ciudad, n_ventas, ticket_prom"
assert vend["n_ventas"].sum() == 2000, "La suma de n_ventas debe ser 2000"
print("Ejercicio 5 correcto ✔")

### Ejercicio 6 · ¿Qué tan grandes son las ventas de cada segmento? *(diagrama de cajas)*
**Contexto.** Para planear el segundo semestre, finanzas quiere ver cómo se distribuye el valor de las ventas por tipo de cliente.

**Tarea.** **Selecciona** solo las ventas del **segundo semestre** (`mes` de 7 a 12) y guárdalas en **`sem2`**. Haz un diagrama de cajas del `total` por `segmento`, **sin mostrar los valores atípicos** (`showfliers=False`).

*Herramientas:* filtro con `>=`, `.boxplot(column=..., by=...)`

In [ ]:
# Ejercicio 6
sem2 = df[df["mes"] >= 7]

ax = sem2.boxplot(
    column="total",
    by="segmento",
    showfliers=False,
    figsize=(8, 4)
)
plt.title("Distribución del total por segmento (segundo semestre)")
plt.suptitle("")
plt.xlabel("segmento")
plt.ylabel("total")
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Verificación
assert len(sem2) == 1050, f"El segundo semestre debe tener 1050 ventas, tienes {len(sem2)}"
print("Ejercicio 6 correcto ✔")

---
# Parte B · Todo en una sola figura (`subplots`)
### Ejercicio 7 · Organiza tus 6 gráficos en una cuadrícula de 2 × 3
**Contexto.** La gerente quiere ver todo en **una sola imagen** para presentarla en la junta.

**Tarea.** Crea una figura con `fig, axs = plt.subplots(2, 3, figsize=(18, 10))` y dibuja cada gráfico en su posición pasando **`ax=axs[fila, columna]`**. Agrega un título general con `fig.suptitle(...)` y termina con `plt.tight_layout()`.

**Importante:** escribe el código en la celda que empieza con `%%writefile figura_base.py` (así el programa guarda tu código en un archivo, que usarás en la Parte C). Luego ejecuta la celda siguiente (`%run -i figura_base.py`) para ver la figura.

*Pistas:* `plot.barh(ax=axs[0, 0])` · `plot.pie(...)` también acepta `ax` · para el boxplot: `boxplot(column=..., by=..., ax=...)` · para la dispersión: `axs[1, 1].scatter(...)`

In [ ]:
%%writefile figura_base.py
fig, axs = plt.subplots(2, 3, figsize=(18, 10))

# (0, 0) - Ingresos por producto (barras horizontales)
df.groupby("producto")["total"].sum().sort_values().plot.barh(
    ax=axs[0, 0], title="Ingresos por producto"
)

# (0, 1) - Ingresos por ciudad (barras)
df.groupby("ciudad")["total"].sum().plot.bar(
    ax=axs[0, 1], title="Ingresos por ciudad"
)

# (0, 2) - Participación por canal (pastel)
df.groupby("canal")["total"].sum().plot.pie(
    ax=axs[0, 2], autopct="%1.1f%%", title="Participación por canal", ylabel=""
)

# (1, 0) - Segmento vs categoría (barras apiladas en %)
tabla_seg_cat = pd.crosstab(df["segmento"], df["categoria"], normalize="index") * 100
tabla_seg_cat.plot.bar(
    ax=axs[1, 0], stacked=True, title="Segmento por categoría (%)"
)

# (1, 1) - Productividad de vendedores (dispersión por ciudad)
vend = (
    df.groupby(["vendedor", "ciudad"])
    .agg(n_ventas=("id_venta", "count"), ticket_prom=("total", "mean"))
    .reset_index()
)
for ciudad in vend["ciudad"].unique():
    vend_ciudad = vend[vend["ciudad"] == ciudad]
    axs[1, 1].scatter(
        vend_ciudad["n_ventas"], vend_ciudad["ticket_prom"], label=ciudad
    )
axs[1, 1].set_title("Productividad de vendedores")
axs[1, 1].legend()

# (1, 2) - Distribución del total por segmento en sem2 (diagrama de cajas)
sem2 = df[df["mes"] >= 7]
sem2.boxplot(
    column="total",
    by="segmento",
    showfliers=False,
    ax=axs[1, 2],
)
axs[1, 2].set_title("Total por segmento (sem2)")

fig.suptitle("Resumen ejecutivo de ventas")
plt.tight_layout()
plt.show()

In [ ]:
%run -i figura_base.py

In [ ]:
# ✅ Verificación
assert len(fig.axes) >= 6, f"La figura debe tener 6 gráficos, tiene {len(fig.axes)}"
print("Ejercicio 7 correcto ✔")

---
# Parte C · Hazla ver bonita con prompts
Ahora **tú escribes el prompt** y se lo envías a Gemini (en [gemini.google.com](https://gemini.google.com) o la IA que prefieras). Gemini **reescribe el código** de tu figura para que se vea profesional, casi como un dashboard. Tú pegas ese código en el notebook y lo ejecutas.

**El ciclo, cada vez:**
1. Ejecuta la celda de herramientas: te muestra **tu código actual** y las **reglas técnicas**.
2. Escribe tu prompt (Ejercicios 8 y 9).
3. En el chat de la IA pega, en este orden: **tu prompt + tu código + las reglas**.
4. Copia **solo el código** de la respuesta (botón *copiar* del bloque) y pégalo en la celda `%%writefile figura_v1.py`. **No incluyas las líneas con \`\`\`** ni el texto que la IA escriba fuera del bloque.
5. Ejecuta la celda y luego `mostrar("v1")` para ver la figura.

### Herramientas *(ya están hechas)*
Ejecuta la celda: define la función `mostrar(nombre)` (dibuja la versión guardada en `figura_<nombre>.py`) e imprime lo que debes copiar a la IA.

In [ ]:
REGLAS_FIGURA = """

Reglas técnicas obligatorias:
- Usa solo pandas y matplotlib (y numpy si lo necesitas). No instales otras librerías.
- Los datos ya están cargados en el DataFrame `df` (columnas: id_venta, fecha, ciudad, canal, segmento, vendedor,
  producto, categoria, metodo_pago, cantidad, precio_unitario, descuento, total, mes). No cargues archivos.
- Conserva los MISMOS 6 gráficos y los mismos datos del código actual; cambia solo la presentación.
- La figura debe quedar en una variable llamada `fig` (2 filas x 3 columnas).
- NO uses plt.show() ni guardes archivos.
- Entrega solo UN bloque de código Python completo.
"""


def mostrar(nombre):
    """Dibuja la versión guardada en figura_<nombre>.py."""
    plt.close("all")
    codigo = open(f"figura_{nombre}.py", encoding="utf-8").read().replace("plt.show()", "")
    exec(codigo, globals())
    display(fig)
    plt.close(fig)


print("=== 1) COPIA ESTE CÓDIGO (tu figura actual) ===")
print(open("figura_base.py", encoding="utf-8").read())
print("\n=== 2) COPIA ESTAS REGLAS al final de tu prompt ===")
print(REGLAS_FIGURA)

### Ejercicio 8 · Tu primer prompt de estilo
Completa las secciones de `prompt_v1`. Sé **específico**: "usa una paleta de azules y naranja" funciona mejor que "hazlo bonito".

- **PÚBLICO Y OBJETIVO:** quién verá la figura y para qué.
- **PALETA Y TIPOGRAFÍA:** colores (por ciudad, por producto…), fondo, tamaños de letra.
- **TÍTULOS:** que cada gráfico tenga un título que diga la **conclusión** (por ejemplo, "Bogotá lidera las ventas"), no solo el nombre.
- **EJES Y NÚMEROS:** formatos con separador de miles o "$ 12 M", etiquetas de ejes claras.
- **LIMPIEZA VISUAL:** quitar bordes y marcos innecesarios, cuadrícula suave, valores sobre las barras.

Ejecuta la primera celda para imprimir tu prompt y cópialo a la IA (con tu código y las reglas). Luego pega el código que te devuelva en la segunda celda y muestra la figura con la tercera.

In [ ]:
prompt_v1 = """
PÚBLICO Y OBJETIVO: Gerencia general para una junta directiva de revisión comercial; debe verse limpio, ejecutivo y fácil de leer de un vistazo.

PALETA Y TIPOGRAFÍA: Usa una paleta profesional basada en tonos azul marino, azul medio, gris pizarra y un color de acento naranja/coral para resaltar el líder o punto clave. Fondo blanco limpio y tipografía clara (título general en 16pt negrita, títulos de cada gráfico en 11pt negrita, etiquetas en 9pt).

TÍTULOS: Cada gráfico debe tener un título que exprese la conclusión principal (por ejemplo: "Cuaderno lidera los ingresos por producto", "Bogotá concentra los mayores ingresos", "Canal telefónico aporta más de la mitad de ventas", "Cada segmento prioriza categorías distintas", "Productividad de vendedores por ciudad" y "Mayorista tiene los tickets más altos en sem2").

EJES Y NÚMEROS: Formatea los valores monetarios grandes en millones (ej. "$ 20 M") o con separador de miles en lugar de notación científica (1e7). Etiqueta claramente los ejes y agrega valores sobre las barras principales cuando ayude a leer rápido.

LIMPIEZA VISUAL: Quita los bordes superior y derecho (spines) de los gráficos, agrega una cuadrícula horizontal o vertical muy suave, elimina el subtítulo automático extra del boxplot y evita que las etiquetas de los ejes se encimen.
"""

print(prompt_v1)
# cópialo en la IA, junto con tu código y las reglas

In [ ]:
%%writefile figura_v1.py
# Pega aquí SOLO el código que te devolvió la IA (sin las líneas ```)
import matplotlib.ticker as mtick

# Paleta ejecutiva: tonos azul marino/pizarra y acento coral/naranja
NAVY = "#1B365D"
BLUE = "#3B6E9C"
SLATE = "#7A8FA6"
LIGHT_SLATE = "#B8C4D0"
CORAL = "#E86F52"
PALETTE_5 = ["#1B365D", "#3B6E9C", "#6897BB", "#9AB8D1", "#E86F52"]
PALETTE_3 = ["#E86F52", "#1B365D", "#6897BB"]

fig, axs = plt.subplots(2, 3, figsize=(18, 10), facecolor="white")
fig.patch.set_facecolor("white")

def estilo_limpio(ax, grid_axis="y"):
    ax.set_facecolor("white")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#CCCCCC")
    ax.spines["bottom"].set_color("#CCCCCC")
    ax.tick_params(colors="#333333", labelsize=9)
    if grid_axis:
        ax.grid(axis=grid_axis, linestyle="--", alpha=0.35, color="#B0B8C1")
        ax.set_axisbelow(True)

fmt_millones = mtick.FuncFormatter(lambda x, _: f"$ {x/1e6:,.1f} M")
fmt_miles = mtick.FuncFormatter(lambda x, _: f"$ {x:,.0f}")

# (0, 0) - Ingresos por producto (barras horizontales)
ing_prod = df.groupby("producto")["total"].sum().sort_values()
colores_prod = [CORAL if i == len(ing_prod) - 1 else BLUE for i in range(len(ing_prod))]
ing_prod.plot.barh(ax=axs[0, 0], color=colores_prod, width=0.72)
axs[0, 0].set_title("Cuaderno lidera los ingresos por producto", fontsize=11, fontweight="bold", color=NAVY, pad=10, loc="left")
axs[0, 0].set_xlabel("Ingresos totales", fontsize=9, color="#333333")
axs[0, 0].set_ylabel("")
axs[0, 0].xaxis.set_major_formatter(fmt_millones)
axs[0, 0].set_xlim(0, ing_prod.max() * 1.18)
for bar in axs[0, 0].patches:
    val = bar.get_width()
    axs[0, 0].text(val + ing_prod.max() * 0.015, bar.get_y() + bar.get_height() / 2,
                   f"$ {val/1e6:.1f}M", va="center", ha="left", fontsize=8, color="#222222")
estilo_limpio(axs[0, 0], grid_axis="x")

# (0, 1) - Ingresos por ciudad (barras)
ing_ciudad = df.groupby("ciudad")["total"].sum()
max_ciudad = ing_ciudad.idxmax()
colores_ciudad = [CORAL if c == max_ciudad else BLUE for c in ing_ciudad.index]
ing_ciudad.plot.bar(ax=axs[0, 1], color=colores_ciudad, width=0.58)
axs[0, 1].set_title("Bogotá concentra los mayores ingresos", fontsize=11, fontweight="bold", color=NAVY, pad=10, loc="left")
axs[0, 1].set_xlabel("")
axs[0, 1].set_ylabel("Ingresos totales", fontsize=9, color="#333333")
axs[0, 1].yaxis.set_major_formatter(fmt_millones)
axs[0, 1].set_ylim(0, ing_ciudad.max() * 1.15)
axs[0, 1].tick_params(axis="x", rotation=0)
for bar in axs[0, 1].patches:
    val = bar.get_height()
    axs[0, 1].text(bar.get_x() + bar.get_width() / 2, val + ing_ciudad.max() * 0.02,
                   f"$ {val/1e6:.1f}M", ha="center", va="bottom", fontsize=8.5, fontweight="bold", color="#222222")
estilo_limpio(axs[0, 1], grid_axis="y")

# (0, 2) - Participación por canal (pastel)
ing_canal = df.groupby("canal")["total"].sum()
colores_canal = [CORAL if c == ing_canal.idxmax() else (NAVY if i == 0 else SLATE) for i, c in enumerate(ing_canal.index)]
wedges, texts, autotexts = axs[0, 2].pie(
    ing_canal,
    labels=ing_canal.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=colores_canal,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    textprops={"fontsize": 9.5, "color": "#222222"}
)
for autotext in autotexts:
    autotext.set_color("white")
    autotext.set_fontweight("bold")
axs[0, 2].set_title("Canal telefónico aporta más de la mitad de ventas", fontsize=11, fontweight="bold", color=NAVY, pad=10, loc="left")
axs[0, 2].set_ylabel("")

# (1, 0) - Segmento vs categoría (barras apiladas en %)
tabla_seg_cat = pd.crosstab(df["segmento"], df["categoria"], normalize="index") * 100
tabla_seg_cat.plot.bar(
    ax=axs[1, 0],
    stacked=True,
    color=PALETTE_5,
    width=0.55,
    edgecolor="white",
    linewidth=0.8
)
axs[1, 0].set_title("Cada segmento prioriza categorías distintas", fontsize=11, fontweight="bold", color=NAVY, pad=10, loc="left")
axs[1, 0].set_xlabel("")
axs[1, 0].set_ylabel("Participación (%)", fontsize=9, color="#333333")
axs[1, 0].set_ylim(0, 105)
axs[1, 0].tick_params(axis="x", rotation=0)
axs[1, 0].legend(title="Categoría", frameon=False, fontsize=8, title_fontsize=8.5, bbox_to_anchor=(1.01, 1), loc="upper left")
estilo_limpio(axs[1, 0], grid_axis="y")

# (1, 1) - Productividad de vendedores (dispersión por ciudad)
vend = (
    df.groupby(["vendedor", "ciudad"])
    .agg(n_ventas=("id_venta", "count"), ticket_prom=("total", "mean"))
    .reset_index()
)
for i, ciudad in enumerate(vend["ciudad"].unique()):
    vend_ciudad = vend[vend["ciudad"] == ciudad]
    axs[1, 1].scatter(
        vend_ciudad["n_ventas"],
        vend_ciudad["ticket_prom"],
        label=ciudad,
        color=PALETTE_5[i % len(PALETTE_5)],
        s=75,
        alpha=0.9,
        edgecolors="white",
        linewidth=0.8
    )
axs[1, 1].set_title("Productividad de vendedores por ciudad", fontsize=11, fontweight="bold", color=NAVY, pad=10, loc="left")
axs[1, 1].set_xlabel("Número de ventas", fontsize=9, color="#333333")
axs[1, 1].set_ylabel("Ticket promedio", fontsize=9, color="#333333")
axs[1, 1].yaxis.set_major_formatter(fmt_miles)
axs[1, 1].legend(title="Ciudad", frameon=False, fontsize=8, title_fontsize=8.5, loc="best")
estilo_limpio(axs[1, 1], grid_axis="both")

# (1, 2) - Distribución del total por segmento en sem2 (diagrama de cajas)
sem2 = df[df["mes"] >= 7]
bp = sem2.boxplot(
    column="total",
    by="segmento",
    showfliers=False,
    ax=axs[1, 2],
    patch_artist=True,
    return_type="dict"
)
for patch, color in zip(bp["total"]["boxes"], [BLUE, CORAL, SLATE]):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
    patch.set_edgecolor(NAVY)
for median in bp["total"]["medians"]:
    median.set_color("white")
    median.set_linewidth(2)
for whisker in bp["total"]["whiskers"]:
    whisker.set_color(NAVY)
for cap in bp["total"]["caps"]:
    cap.set_color(NAVY)

fig.suptitle("Resumen ejecutivo de ventas — Desempeño comercial", fontsize=16, fontweight="bold", color=NAVY, y=0.98)
axs[1, 2].set_title("Mayorista tiene los tickets más altos en sem2", fontsize=11, fontweight="bold", color=NAVY, pad=10, loc="left")
axs[1, 2].set_xlabel("")
axs[1, 2].set_ylabel("Valor de venta", fontsize=9, color="#333333")
axs[1, 2].yaxis.set_major_formatter(fmt_miles)
estilo_limpio(axs[1, 2], grid_axis="y")

plt.tight_layout(rect=[0, 0, 1, 0.94])

In [ ]:
mostrar("v1")

### Ejercicio 9 · Itera: mejora tu prompt
Mira la figura que obtuviste: ¿qué no te gusta? Agrega una sección **AJUSTES** con **al menos dos cambios concretos** (otra paleta, letras más grandes, una dona en lugar del pie, anotar el valor máximo del gráfico de líneas…). Es la **iteración de prompts** que vimos en clase. Repite el ciclo: copia `prompt_v2` a la IA (con tu código base y las reglas), pega el código en `figura_v2.py` y muestra la figura.

In [ ]:
prompt_v2 = prompt_v1 + """
AJUSTES:
1. Cambia la paleta de colores a un estilo ejecutivo de alta dirección: azul medianoche oscuro (#0F2537), gris pizarra (#4A607A), gris cálido (#9AA5B1) y dorado metálico/ámbar (#C59B27) como color de acento principal para destacar los líderes, junto con un fondo ligeramente marfil/gris muy tenue (#FAFAFA).
2. Convierte el gráfico de participación por canal en un gráfico de dona (donut chart) con el total general en el centro, y aumenta ligeramente el tamaño de las etiquetas de datos para mejorar la lectura en pantalla.
"""

print(prompt_v2)

In [ ]:
%%writefile figura_v2.py
# Pega aquí el código que te devolvió la IA para la versión 2
import matplotlib.ticker as mtick

# Paleta ejecutiva sobria con acento dorado
MIDNIGHT = "#0F2537"
SLATE_DARK = "#2C4A6E"
SLATE_MED = "#5C768D"
WARM_GRAY = "#9AA5B1"
GOLD = "#C59B27"
GOLD_SOFT = "#DFBE5F"
BG_PAGE = "#FAFAFA"

PALETTE_5 = ["#0F2537", "#2C4A6E", "#5C768D", "#9AA5B1", "#C59B27"]

fig, axs = plt.subplots(2, 3, figsize=(18, 10), facecolor=BG_PAGE)
fig.patch.set_facecolor(BG_PAGE)

def estilo_ejecutivo(ax, grid_axis="y"):
    ax.set_facecolor("white")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#D0D5DD")
    ax.spines["bottom"].set_color("#D0D5DD")
    ax.tick_params(colors="#222222", labelsize=9.5)
    if grid_axis:
        ax.grid(axis=grid_axis, linestyle="--", alpha=0.4, color="#CBD2D9")
        ax.set_axisbelow(True)

fmt_millones = mtick.FuncFormatter(lambda x, _: f"$ {x/1e6:,.1f} M")
fmt_miles = mtick.FuncFormatter(lambda x, _: f"$ {x:,.0f}")

# (0, 0) - Ingresos por producto (barras horizontales)
ing_prod = df.groupby("producto")["total"].sum().sort_values()
colores_prod = [GOLD if i == len(ing_prod) - 1 else SLATE_DARK for i in range(len(ing_prod))]
ing_prod.plot.barh(ax=axs[0, 0], color=colores_prod, width=0.72)
axs[0, 0].set_title("Cuaderno lidera los ingresos por producto", fontsize=11.5, fontweight="bold", color=MIDNIGHT, pad=10, loc="left")
axs[0, 0].set_xlabel("Ingresos totales", fontsize=9.5, color="#333333")
axs[0, 0].set_ylabel("")
axs[0, 0].xaxis.set_major_formatter(fmt_millones)
axs[0, 0].set_xlim(0, ing_prod.max() * 1.20)
for i, bar in enumerate(axs[0, 0].patches):
    val = bar.get_width()
    es_lider = (i == len(ing_prod) - 1)
    axs[0, 0].text(
        val + ing_prod.max() * 0.015,
        bar.get_y() + bar.get_height() / 2,
        f"$ {val/1e6:.1f}M",
        va="center",
        ha="left",
        fontsize=8.5,
        fontweight="bold" if es_lider else "normal",
        color=MIDNIGHT if es_lider else "#333333"
    )
estilo_ejecutivo(axs[0, 0], grid_axis="x")

# (0, 1) - Ingresos por ciudad (barras)
ing_ciudad = df.groupby("ciudad")["total"].sum()
max_ciudad = ing_ciudad.idxmax()
colores_ciudad = [GOLD if c == max_ciudad else SLATE_DARK for c in ing_ciudad.index]
ing_ciudad.plot.bar(ax=axs[0, 1], color=colores_ciudad, width=0.55)
axs[0, 1].set_title("Bogotá concentra los mayores ingresos", fontsize=11.5, fontweight="bold", color=MIDNIGHT, pad=10, loc="left")
axs[0, 1].set_xlabel("")
axs[0, 1].set_ylabel("Ingresos totales", fontsize=9.5, color="#333333")
axs[0, 1].yaxis.set_major_formatter(fmt_millones)
axs[0, 1].set_ylim(0, ing_ciudad.max() * 1.16)
axs[0, 1].tick_params(axis="x", rotation=0)
for bar in axs[0, 1].patches:
    val = bar.get_height()
    axs[0, 1].text(
        bar.get_x() + bar.get_width() / 2,
        val + ing_ciudad.max() * 0.02,
        f"$ {val/1e6:.1f}M",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        color=MIDNIGHT
    )
estilo_ejecutivo(axs[0, 1], grid_axis="y")

# (0, 2) - Participación por canal (gráfico de dona con acento dorado)
ing_canal = df.groupby("canal")["total"].sum()
colores_canal = [GOLD if c == ing_canal.idxmax() else (MIDNIGHT if i == 0 else SLATE_MED) for i, c in enumerate(ing_canal.index)]
wedges, texts, autotexts = axs[0, 2].pie(
    ing_canal,
    labels=ing_canal.index,
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.76,
    colors=colores_canal,
    wedgeprops={"width": 0.42, "edgecolor": "white", "linewidth": 2.5},
    textprops={"fontsize": 10, "color": MIDNIGHT, "fontweight": "bold"}
)
for autotext in autotexts:
    autotext.set_color("white")
    autotext.set_fontsize(9.5)
    autotext.set_fontweight("bold")
axs[0, 2].text(0, 0, f"Total\n$ {ing_canal.sum()/1e6:.1f}M", ha="center", va="center", fontsize=11, fontweight="bold", color=MIDNIGHT)
axs[0, 2].set_title("Canal telefónico aporta más de la mitad de ventas", fontsize=11.5, fontweight="bold", color=MIDNIGHT, pad=10, loc="left")
axs[0, 2].set_ylabel("")

# (1, 0) - Segmento vs categoría (barras apiladas en %)
tabla_seg_cat = pd.crosstab(df["segmento"], df["categoria"], normalize="index") * 100
tabla_seg_cat.plot.bar(
    ax=axs[1, 0],
    stacked=True,
    color=PALETTE_5,
    width=0.52,
    edgecolor="white",
    linewidth=1
)
axs[1, 0].set_title("Cada segmento prioriza categorías distintas", fontsize=11.5, fontweight="bold", color=MIDNIGHT, pad=10, loc="left")
axs[1, 0].set_xlabel("")
axs[1, 0].set_ylabel("Participación (%)", fontsize=9.5, color="#333333")
axs[1, 0].set_ylim(0, 105)
axs[1, 0].tick_params(axis="x", rotation=0)
axs[1, 0].legend(title="Categoría", frameon=False, fontsize=8.5, title_fontsize=9, bbox_to_anchor=(1.01, 1), loc="upper left")
estilo_ejecutivo(axs[1, 0], grid_axis="y")

# (1, 1) - Productividad de vendedores (dispersión por ciudad)
vend = (
    df.groupby(["vendedor", "ciudad"])
    .agg(n_ventas=("id_venta", "count"), ticket_prom=("total", "mean"))
    .reset_index()
)
for i, ciudad in enumerate(vend["ciudad"].unique()):
    vend_ciudad = vend[vend["ciudad"] == ciudad]
    axs[1, 1].scatter(
        vend_ciudad["n_ventas"],
        vend_ciudad["ticket_prom"],
        label=ciudad,
        color=PALETTE_5[i % len(PALETTE_5)],
        s=90,
        alpha=0.92,
        edgecolors="white",
        linewidth=1
    )
axs[1, 1].set_title("Productividad de vendedores por ciudad", fontsize=11.5, fontweight="bold", color=MIDNIGHT, pad=10, loc="left")
axs[1, 1].set_xlabel("Número de ventas", fontsize=9.5, color="#333333")
axs[1, 1].set_ylabel("Ticket promedio", fontsize=9.5, color="#333333")
axs[1, 1].yaxis.set_major_formatter(fmt_miles)
axs[1, 1].legend(title="Ciudad", frameon=False, fontsize=8.5, title_fontsize=9, loc="best")
estilo_ejecutivo(axs[1, 1], grid_axis="both")

# (1, 2) - Distribución del total por segmento en sem2 (diagrama de cajas)
sem2 = df[df["mes"] >= 7]
bp = sem2.boxplot(
    column="total",
    by="segmento",
    showfliers=False,
    ax=axs[1, 2],
    patch_artist=True,
    return_type="dict"
)
for patch, color in zip(bp["total"]["boxes"], [SLATE_DARK, GOLD, WARM_GRAY]):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)
    patch.set_edgecolor(MIDNIGHT)
    patch.set_linewidth(1.2)
for median in bp["total"]["medians"]:
    median.set_color("white")
    median.set_linewidth(2.2)
for whisker in bp["total"]["whiskers"]:
    whisker.set_color(MIDNIGHT)
for cap in bp["total"]["caps"]:
    cap.set_color(MIDNIGHT)

fig.suptitle("Resumen ejecutivo de ventas — Desempeño comercial", fontsize=16, fontweight="bold", color=MIDNIGHT, y=0.98)
axs[1, 2].set_title("Mayorista tiene los tickets más altos en sem2", fontsize=11.5, fontweight="bold", color=MIDNIGHT, pad=10, loc="left")
axs[1, 2].set_xlabel("")
axs[1, 2].set_ylabel("Valor de venta", fontsize=9.5, color="#333333")
axs[1, 2].yaxis.set_major_formatter(fmt_miles)
estilo_ejecutivo(axs[1, 2], grid_axis="y")

plt.tight_layout(rect=[0, 0, 1, 0.94])

In [ ]:
mostrar("v2")

### Ejercicio 10 *(opcional)* · Una tercera versión
Si quieres seguir mejorando, crea `prompt_v3` (parte de `prompt_v2`), repite el ciclo con `figura_v3.py` y muestra `mostrar("v3")`.

In [ ]:
prompt_v3 = prompt_v2 + """
AJUSTES FINALES:
1. Agrega un subtítulo general debajo del título principal con el resumen del periodo analizado y el total de transacciones.
2. En el gráfico de barras apiladas (segmento por categoría), escribe el porcentaje dentro de cada segmento de barra (cuando sea >= 10%) para facilitar la lectura.
3. En el gráfico de productividad de vendedores (dispersión), agrega líneas punteadas con el promedio de ventas y ticket promedio, y etiqueta directamente al vendedor con mayor ticket promedio.
"""

print(prompt_v3)

In [ ]:
%%writefile figura_v3.py
import matplotlib.ticker as mtick

# Paleta ejecutiva sobria con acento dorado
MIDNIGHT = "#0F2537"
SLATE_DARK = "#2C4A6E"
SLATE_MED = "#5C768D"
WARM_GRAY = "#9AA5B1"
GOLD = "#C59B27"
BG_PAGE = "#FAFAFA"

PALETTE_5 = ["#0F2537", "#2C4A6E", "#5C768D", "#9AA5B1", "#C59B27"]

fig, axs = plt.subplots(2, 3, figsize=(18, 10.5), facecolor=BG_PAGE)
fig.patch.set_facecolor(BG_PAGE)

def estilo_ejecutivo(ax, grid_axis="y"):
    ax.set_facecolor("white")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#D0D5DD")
    ax.spines["bottom"].set_color("#D0D5DD")
    ax.tick_params(colors="#222222", labelsize=9.5)
    if grid_axis:
        ax.grid(axis=grid_axis, linestyle="--", alpha=0.4, color="#CBD2D9")
        ax.set_axisbelow(True)

fmt_millones = mtick.FuncFormatter(lambda x, _: f"$ {x/1e6:,.1f} M")
fmt_miles = mtick.FuncFormatter(lambda x, _: f"$ {x:,.0f}")

# (0, 0) - Ingresos por producto (barras horizontales)
ing_prod = df.groupby("producto")["total"].sum().sort_values()
colores_prod = [GOLD if i == len(ing_prod) - 1 else SLATE_DARK for i in range(len(ing_prod))]
ing_prod.plot.barh(ax=axs[0, 0], color=colores_prod, width=0.72)
axs[0, 0].set_title("Cuaderno lidera los ingresos por producto", fontsize=11.5, fontweight="bold", color=MIDNIGHT, pad=10, loc="left")
axs[0, 0].set_xlabel("Ingresos totales", fontsize=9.5, color="#333333")
axs[0, 0].set_ylabel("")
axs[0, 0].xaxis.set_major_formatter(fmt_millones)
axs[0, 0].set_xlim(0, ing_prod.max() * 1.22)
for i, bar in enumerate(axs[0, 0].patches):
    val = bar.get_width()
    es_lider = (i == len(ing_prod) - 1)
    axs[0, 0].text(
        val + ing_prod.max() * 0.015,
        bar.get_y() + bar.get_height() / 2,
        f"$ {val/1e6:.1f}M",
        va="center",
        ha="left",
        fontsize=8.5,
        fontweight="bold" if es_lider else "normal",
        color=MIDNIGHT if es_lider else "#333333"
    )
estilo_ejecutivo(axs[0, 0], grid_axis="x")

# (0, 1) - Ingresos por ciudad (barras)
ing_ciudad = df.groupby("ciudad")["total"].sum()
max_ciudad = ing_ciudad.idxmax()
colores_ciudad = [GOLD if c == max_ciudad else SLATE_DARK for c in ing_ciudad.index]
ing_ciudad.plot.bar(ax=axs[0, 1], color=colores_ciudad, width=0.55)
axs[0, 1].set_title("Bogotá concentra los mayores ingresos", fontsize=11.5, fontweight="bold", color=MIDNIGHT, pad=10, loc="left")
axs[0, 1].set_xlabel("")
axs[0, 1].set_ylabel("Ingresos totales", fontsize=9.5, color="#333333")
axs[0, 1].yaxis.set_major_formatter(fmt_millones)
axs[0, 1].set_ylim(0, ing_ciudad.max() * 1.16)
axs[0, 1].tick_params(axis="x", rotation=0)
for bar in axs[0, 1].patches:
    val = bar.get_height()
    axs[0, 1].text(
        bar.get_x() + bar.get_width() / 2,
        val + ing_ciudad.max() * 0.02,
        f"$ {val/1e6:.1f}M",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        color=MIDNIGHT
    )
estilo_ejecutivo(axs[0, 1], grid_axis="y")

# (0, 2) - Participación por canal (gráfico de dona con acento dorado)
ing_canal = df.groupby("canal")["total"].sum()
colores_canal = [GOLD if c == ing_canal.idxmax() else (MIDNIGHT if i == 0 else SLATE_MED) for i, c in enumerate(ing_canal.index)]
wedges, texts, autotexts = axs[0, 2].pie(
    ing_canal,
    labels=ing_canal.index,
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.76,
    colors=colores_canal,
    wedgeprops={"width": 0.42, "edgecolor": "white", "linewidth": 2.5},
    textprops={"fontsize": 10, "color": MIDNIGHT, "fontweight": "bold"}
)
for autotext in autotexts:
    autotext.set_color("white")
    autotext.set_fontsize(9.5)
    autotext.set_fontweight("bold")
axs[0, 2].text(0, 0, f"Total\n$ {ing_canal.sum()/1e6:.1f}M", ha="center", va="center", fontsize=11, fontweight="bold", color=MIDNIGHT)
axs[0, 2].set_title("Canal telefónico aporta más de la mitad de ventas", fontsize=11.5, fontweight="bold", color=MIDNIGHT, pad=10, loc="left")
axs[0, 2].set_ylabel("")

# (1, 0) - Segmento vs categoría (barras apiladas con porcentajes internos)
tabla_seg_cat = pd.crosstab(df["segmento"], df["categoria"], normalize="index") * 100
tabla_seg_cat.plot.bar(
    ax=axs[1, 0],
    stacked=True,
    color=PALETTE_5,
    width=0.52,
    edgecolor="white",
    linewidth=1
)
for container in axs[1, 0].containers:
    labels = [f"{v:.0f}%" if v >= 10 else "" for v in container.datavalues]
    axs[1, 0].bar_label(container, labels=labels, label_type="center", fontsize=8, fontweight="bold", color="white")

axs[1, 0].set_title("Cada segmento prioriza categorías distintas", fontsize=11.5, fontweight="bold", color=MIDNIGHT, pad=10, loc="left")
axs[1, 0].set_xlabel("")
axs[1, 0].set_ylabel("Participación (%)", fontsize=9.5, color="#333333")
axs[1, 0].set_ylim(0, 105)
axs[1, 0].tick_params(axis="x", rotation=0)
axs[1, 0].legend(title="Categoría", frameon=False, fontsize=8.5, title_fontsize=9, bbox_to_anchor=(1.01, 1), loc="upper left")
estilo_ejecutivo(axs[1, 0], grid_axis="y")

# (1, 1) - Productividad de vendedores (dispersión con promedios y destacado)
vend = (
    df.groupby(["vendedor", "ciudad"])
    .agg(n_ventas=("id_venta", "count"), ticket_prom=("total", "mean"))
    .reset_index()
)
for i, ciudad in enumerate(vend["ciudad"].unique()):
    vend_ciudad = vend[vend["ciudad"] == ciudad]
    axs[1, 1].scatter(
        vend_ciudad["n_ventas"],
        vend_ciudad["ticket_prom"],
        label=ciudad,
        color=PALETTE_5[i % len(PALETTE_5)],
        s=95,
        alpha=0.92,
        edgecolors="white",
        linewidth=1
    )

axs[1, 1].axvline(vend["n_ventas"].mean(), color=WARM_GRAY, linestyle=":", linewidth=1.2)
axs[1, 1].axhline(vend["ticket_prom"].mean(), color=WARM_GRAY, linestyle=":", linewidth=1.2)

top_ticket = vend.loc[vend["ticket_prom"].idxmax()]
axs[1, 1].annotate(
    f"Mayor ticket: {top_ticket['vendedor']}",
    xy=(top_ticket["n_ventas"], top_ticket["ticket_prom"]),
    xytext=(10, -4),
    textcoords="offset points",
    fontsize=8.5,
    fontweight="bold",
    color=MIDNIGHT
)

axs[1, 1].set_title("Productividad de vendedores por ciudad", fontsize=11.5, fontweight="bold", color=MIDNIGHT, pad=10, loc="left")
axs[1, 1].set_xlabel("Número de ventas", fontsize=9.5, color="#333333")
axs[1, 1].set_ylabel("Ticket promedio", fontsize=9.5, color="#333333")
axs[1, 1].yaxis.set_major_formatter(fmt_miles)
axs[1, 1].legend(title="Ciudad", frameon=False, fontsize=8.5, title_fontsize=9, loc="lower right")
estilo_ejecutivo(axs[1, 1], grid_axis="both")

# (1, 2) - Distribución del total por segmento en sem2 (diagrama de cajas)
sem2 = df[df["mes"] >= 7]
bp = sem2.boxplot(
    column="total",
    by="segmento",
    showfliers=False,
    ax=axs[1, 2],
    patch_artist=True,
    return_type="dict"
)
for patch, color in zip(bp["total"]["boxes"], [SLATE_DARK, GOLD, WARM_GRAY]):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)
    patch.set_edgecolor(MIDNIGHT)
    patch.set_linewidth(1.2)
for median in bp["total"]["medians"]:
    median.set_color("white")
    median.set_linewidth(2.2)
for whisker in bp["total"]["whiskers"]:
    whisker.set_color(MIDNIGHT)
for cap in bp["total"]["caps"]:
    cap.set_color(MIDNIGHT)

fig.suptitle("Resumen ejecutivo de ventas — Desempeño comercial", fontsize=16, fontweight="bold", color=MIDNIGHT, y=0.985)
fig.text(0.5, 0.945, f"Análisis integral sobre {len(df):,} transacciones · Ingresos totales: $ {df['total'].sum()/1e6:.1f} M",
         ha="center", fontsize=10.5, color="#4A5568")

axs[1, 2].set_title("Mayorista tiene los tickets más altos en sem2", fontsize=11.5, fontweight="bold", color=MIDNIGHT, pad=10, loc="left")
axs[1, 2].set_xlabel("")
axs[1, 2].set_ylabel("Valor de venta", fontsize=9.5, color="#333333")
axs[1, 2].yaxis.set_major_formatter(fmt_miles)
estilo_ejecutivo(axs[1, 2], grid_axis="y")

plt.tight_layout(rect=[0, 0, 1, 0.92])

In [ ]:
mostrar("v3")

In [ ]:
# ✅ Verificación
import os
assert os.path.exists("figura_v1.py") and os.path.exists("figura_v2.py"), "Faltan figura_v1.py y figura_v2.py: pega el código de cada versión"
assert "ESCRIBE AQUÍ" not in prompt_v1 and prompt_v2 != prompt_v1 and "ESCRIBE AQUÍ" not in prompt_v2, "Completa tus prompts v1 y v2"
print("Parte C correcta ✔")

---
# Parte D · Elige tu favorita y publícala
### Ejercicio 11 · Tu dashboard final y tus conclusiones
Mira tus versiones y **elige la que más te guste**. Luego escribe **3 hallazgos**: frases cortas con **cifras reales** sacadas de tus tablas. Se mostrarán debajo de tu dashboard.

*Ejemplo de hallazgo:* "Bogotá genera el 32 % de los ingresos de la cadena."

In [ ]:
figura_elegida = "v3"    # cambia por "v1", "v2" o "v3"

hallazgos = [
    "El canal telefónico concentra el 51.8 % del total de ingresos de la compañía.",
    "Bogotá lidera la facturación regional superando los $ 37 M, seguida por Medellín.",
    "Cuaderno es el producto con mayor volumen de ingresos, rebasando los $ 20 M.",
]

In [ ]:
# ✅ Verificación
import os
assert os.path.exists(f"figura_{figura_elegida}.py"), f"No existe figura_{figura_elegida}.py: revisa figura_elegida"
assert len(hallazgos) == 3 and all(len(h) > 25 and "ESCRIBE AQUÍ" not in h for h in hallazgos), "Escribe tus 3 hallazgos con cifras"
print("Ejercicio 11 correcto ✔")

### Tu dashboard final
Ejecuta la celda: muestra la versión que elegiste y tus tres hallazgos. **Esta es la salida que se verá en GitHub.**

In [ ]:
mostrar(figura_elegida)

print("Hallazgos:")
for h in hallazgos:
    print("•", h)

### Ejercicio 12 · Publica tu notebook en GitHub
GitHub muestra los notebooks **con sus figuras**, así que tu notebook publicado es tu dashboard. Solo necesitas una cuenta gratuita en [github.com](https://github.com).

**Antes de publicar:**
- Ejecuta todo el notebook de arriba abajo (*Entorno de ejecución → Ejecutar todo*) para que las figuras queden guardadas en las salidas.
- **Nunca escribas tu API key dentro de una celda.** Los *Secrets* de Colab no se guardan en el notebook y por eso son seguros; el repositorio será público.

**Pasos:**
1. **Crea el repositorio:** botón **New** → nombre `mi-dashboard-ventas` → **Public** → *Create repository*.
2. **Guarda el notebook en GitHub.** Opción A: en Colab, *Archivo → Guardar una copia en GitHub*, autoriza, elige tu repositorio y acepta. Opción B: *Archivo → Descargar → .ipynb* y, en el repositorio, **Add file → Upload files** y arrastra el archivo.
3. **Abre el notebook en el repositorio** (haz clic en el archivo `.ipynb`): GitHub lo dibuja con tus figuras.
4. **Copia el enlace** de la barra del navegador. Debe verse así: `https://github.com/TU-USUARIO/mi-dashboard-ventas/blob/main/NOMBRE.ipynb`
5. **Compruébalo:** ábrelo en una ventana de incógnito. Si lo ves sin iniciar sesión, está público.

**Si no se ven las figuras:** vuelve a Colab, ejecuta todo y guarda de nuevo. Si GitHub dice *Unable to render code block*, recarga la página unos segundos después.

---
**Para entregar:** (1) el **enlace público** de tu notebook en GitHub, con tu dashboard final visible, todos los integrantes deben estar como colaboradores en el repositorio, y (2) tus prompts (`prompt_v1`, `prompt_v2` y, si la hiciste, `prompt_v3`) más **tres líneas** sobre qué tuviste que corregir de lo que hizo Gemini y cómo lo pediste (escríbelas en una celda de texto al final del notebook) (3) El notebook (ipynb) con los nombres de todos los integrantes.